# From 300 leaflets to a table you can merge

Session 4, Part 5. The professions de foi of the first round of the **1978 legislative election**: Archelec (Sciences Po, CEVIPOF), through the Internet Archive API. Who are the candidates? An LLM fills in a form for each leaflet, Claude and GPT, and we check every field against the catalogue Sciences Po wrote by hand.

| Part | What you do | What you get |
| --- | --- | --- |
| 1 | Collect 300 documents and their catalogue records | the OCR texts, and the candidates as Sciences Po recorded them |
| 2 | An LLM fills in a form for each document, Claude and GPT | a table, checked field by field against the catalogue |
| 3 | Keep what a referee would ask for | the table and a bundle to download |

**Before you start.** Colab Secrets (key icon, left sidebar): `ANTHROPIC_API_KEY` and `OPENAI_API_KEY`, "Notebook access" on. Never paste a key into a cell. A CPU runtime is enough.

The texts are under a restrictive licence (CC BY-NC-ND): keep them out of any public repository.

The pages were read in 2021 by a classical OCR (Tesseract). How to read a scan again, and compare: the other notebook of today, `s4_ocr.ipynb`.

## 0. Setup (run once; nothing to explain here)

In [ ]:
%pip install -q anthropic openai pydantic

In [ ]:
import datetime, hashlib, json, os, pathlib, random, re, shutil, time, unicodedata
from concurrent.futures import ThreadPoolExecutor
from typing import Literal

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from pydantic import BaseModel, ConfigDict

# ---- Parameters: everything a referee would ask about is here ----
DATE = "1978-03-12"            # first round of the 1978 legislative election (4,010 professions de foi in Archelec)
N = 300                        # documents drawn at random from that election. Keep N and SEED as they are:
SEED = 2026                    # the draw: fixed, so that everyone has the same 300 documents
CLAUDE_MODEL = "claude-haiku-5-5"
GPT_MODEL = "gpt-6-luna"
MAX_CHARS = 15000              # longer texts are cut before they go to a model (the count is printed)
PRICE = {"in": 0.10, "out": 0.50, "cache_read": 0.01, "cache_write": 0.125}  # $ per million tokens, both models, Nov. 2026

RUN_ID = datetime.datetime.now().strftime("%Y%m%d-%H%M")
ROOT = pathlib.Path("s4_work")
RAW = ROOT / "data" / "raw" / "archelec" / DATE     # raw answers of the API, never edited
LLM = ROOT / "outputs" / "llm"                       # every model answer, appended as it arrives
RUN = ROOT / "outputs" / RUN_ID
for p in (RAW / "texts", LLM, RUN):
    p.mkdir(parents=True, exist_ok=True)
print("Run:", RUN)

In [ ]:
# Plumbing: keys, requests with retries, text normalisation, scores. Read it once if you like.

def get_secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        pass
    if os.environ.get(name):
        return os.environ[name]
    for f in (pathlib.Path(".env"), pathlib.Path.home() / ".env"):
        if f.exists():
            for line in f.read_text().splitlines():
                if line.strip().startswith(name + "="):
                    return line.split("=", 1)[1].strip().strip('"').strip("'")
    return None


HEADERS = {"User-Agent": "uzh-llm-socsci-2026 (course notebook)"}


def get(url, params=None, tries=5):
    """GET with retries. Raises after the last try: a silent failure is the worst kind."""
    for i in range(tries):
        try:
            r = requests.get(url, params=params, headers=HEADERS, timeout=60)
            if r.status_code == 200:
                return r
            if r.status_code == 404:
                return None
        except requests.RequestException:
            pass
        time.sleep(2 ** i)
    raise RuntimeError(f"failed after {tries} tries: {url}")


def fetch(identifier):
    """The catalogue record and the OCR text of one document. Rerun-safe: texts on disk are not downloaded again."""
    txt = RAW / "texts" / f"{identifier}.txt"
    m = get(f"https://archive.org/metadata/{identifier}")
    meta = (m.json().get("metadata") or None) if m is not None else None
    if not txt.exists():
        r = get(f"https://archive.org/download/{identifier}/{identifier}_djvu.txt")
        if r is not None:
            txt.write_text(r.text, encoding="utf-8")
    return identifier, meta


def norm(s):
    """Lower case, no accents, no punctuation: 'Député-Maire' -> 'depute maire'."""
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode().lower()
    s = re.sub(r"[’`]", "'", s)
    return re.sub(r"\s+", " ", re.sub(r"[^a-z0-9' ]", " ", s)).strip()


EMPTY = {"", "nr", "non precise", "none", "nan", "?"}   # "NR" = not recorded in the catalogue


def empty(v):
    return norm(v) in EMPTY


def flat(v):
    return " / ".join(map(str, v)) if isinstance(v, list) else ("" if v is None else str(v))


print("ready")

## 1. Collect: a data API, again

Internet Archive's search API. The same four questions as in Session 3: what did I ask for, what came back, what is missing, and against which number do I check?

In [ ]:
QUERY = f'collection:archiveselectoralesducevipof AND date:{DATE} AND type:"profession de foi"'
r = get("https://archive.org/advancedsearch.php",
        {"q": QUERY, "fl[]": "identifier", "sort[]": "identifier asc", "rows": 10000, "output": "json"})
FOUND = r.json()["response"]
FOUND["numFound"], len(FOUND["docs"])

Check 1: the search returned every document it found. If not, it is paginating, and we read page one only.

In [ ]:
ALL_IDS = [d["identifier"] for d in FOUND["docs"]]
assert len(ALL_IDS) == FOUND["numFound"], "the search is paginating: ask for more rows"
(RAW / "ids.json").write_text(json.dumps({"query": QUERY, "numFound": FOUND["numFound"], "ids": ALL_IDS}))
ALL_IDS[:3]

An identifier reads like a code: election, year, month, *département*, constituency, round, document. A random draw of `N`, fixed by `SEED`.

In [ ]:
SAMPLE = sorted(random.Random(SEED).sample(ALL_IDS, N))
len(SAMPLE), SAMPLE[0]

In [ ]:
with ThreadPoolExecutor(8) as pool:
    FETCHED = dict(pool.map(fetch, SAMPLE))
TEXT = {i: (RAW / "texts" / f"{i}.txt").read_text(encoding="utf-8")
        for i in SAMPLE if (RAW / "texts" / f"{i}.txt").exists()}
missing = [i for i in SAMPLE if FETCHED.get(i) is None or i not in TEXT]
print(f"requested {len(SAMPLE)} = returned {len(SAMPLE) - len(missing)} + missing {len(missing)}", missing[:5])

Check 2 and 3: no empty text, no duplicate. An OCR that failed gives a text of a few characters.

In [ ]:
lengths = pd.Series({i: len(t) for i, t in TEXT.items()})
short = lengths[lengths < 300].index.tolist()
dupes = pd.Series({i: hashlib.md5(t.encode()).hexdigest() for i, t in TEXT.items()}).duplicated().sum()
print("characters per text:", lengths.describe()[["min", "50%", "max"]].round().to_dict(),
      "| shorter than 300:", len(short), "| duplicates:", dupes, "| longer than MAX_CHARS:", int((lengths > MAX_CHARS).sum()))

One document. Open the PDF next to it: what does the OCR get wrong?

In [ ]:
print(f"https://archive.org/details/{SAMPLE[0]}\n")
print(TEXT[SAMPLE[0]][:1500])

### The catalogue: someone else's coding

Every document has a record written by Sciences Po: the candidate, the substitute (*suppléant*), the profession, the current offices, the party. "NR" means *not recorded*. This is the number we did not produce: Part 2 checks the LLM against it.

In [ ]:
FIELDS = {"nom": "ba-nom-titulaire", "prenom": "bb-prenom-titulaire", "sexe": "bc-sexe-titulaire",
          "profession": "be-profession-titulaire", "mandat": "bf-mandat-en-cours-titulaire",
          "soutien": "bj-soutien-titulaire", "liste": "bk-liste-titulaire",
          "supp_nom": "ca-nom-suppleant", "supp_profession": "ce-profession-suppleant",
          "departement": "departement", "circonscription": "df-circonscription"}
CAT = pd.DataFrame({i: {k: flat(m.get(v)) for k, v in FIELDS.items()} for i, m in FETCHED.items() if m}).T
CAT.head()

**X, the explanatory variable**: does the candidate hold a local office now (mayor, deputy mayor, municipal, general or regional councillor)? From the catalogue. Sitting deputies are flagged separately.

In [ ]:
LOCAL = re.compile(r"\bmaire\b|\badjoint|conseil\w* (municipal|general|regional|departemental|d'arrondissement|de paris)"
                   r"|president du conseil (general|regional|departemental)")


def is_local(m):
    return int(bool(LOCAL.search(norm(m))))


def is_deputy(m):
    return int(bool(re.search(r"\bdepute", norm(m))))


CAT["local_mandate"] = CAT.mandat.map(is_local)
CAT["deputy"] = CAT.mandat.map(is_deputy)
KEEP = [i for i in SAMPLE if i in TEXT and i in CAT.index and i not in short]
CAT = CAT.loc[KEEP]
print(len(KEEP), "documents kept |", CAT.local_mandate.mean().round(2), "hold a local office |",
      CAT.deputy.mean().round(2), "are sitting deputies")
CAT.mandat[CAT.local_mandate == 1].value_counts().head(8)

## 2. Extracting with an LLM: from a text to a table

Classification gives one label per document. **Extraction** fills in a form: who is the candidate, what is their profession, which offices do they hold, who supports them, who is the substitute. Plus one box that is a judgement, not a copy: `local_offer`: does the leaflet make a promise about a specific place? Session 5 comes back to it.

The form is a schema: the model cannot answer outside it. An empty field is an answer; an invented one is an error.

In [ ]:
class Person(BaseModel):
    model_config = ConfigDict(extra="forbid")
    nom: str
    prenom: str
    sexe: Literal["H", "F", "?"]
    profession: str
    mandats_en_cours: list[str]


class LocalOffer(BaseModel):
    model_config = ConfigDict(extra="forbid")
    quote: str          # the evidence first, then the decision
    present: bool


class Form(BaseModel):
    model_config = ConfigDict(extra="forbid")
    titulaire: Person
    soutien: str
    suppleant: Person
    local_offer: LocalOffer

In [ ]:
INSTRUCTIONS = """You read the campaign statement (profession de foi) of one candidate in a French legislative election, from an OCR scan. Fill in the form. The French text is authoritative. It may contain OCR errors: read through them, but never guess what cannot be read. Leave a field empty rather than guess: an empty field is an answer, an invented value is an error.

titulaire: the candidate the document is for.
suppleant: the candidate's substitute (suppléant, remplaçant). If the document names none, leave all its fields empty ("" and an empty list) and set sexe to "?".

For each person:
- nom, prenom: the surname and first name as printed, without titles (M., Mme, Docteur, Maître).
- sexe: "H" or "F" when the text shows it (Madame, Monsieur, a feminine or masculine form, the first name); "?" if you cannot tell.
- profession: the occupation as printed, for example "agriculteur", "ouvrier", "professeur", "médecin". Not a party position, not an elected office. "" if none is given.
- mandats_en_cours: every elected office the person holds at the time of the election, as printed, for example "maire de Carvin", "conseiller général du canton de Lens", "député sortant". Not past offices ("ancien maire"), not party, union or association positions. An empty list if none.

soutien: the party, movement or coalition that presents or supports the candidate, as printed, for example "Parti communiste français", "Union pour la démocratie française", "RPR". "" if none is named.

local_offer: a geographically specific local issue, institution, project, service, firm, infrastructure, place, or population. A candidate name, constituency heading, or place name alone is insufficient.
- quote: one verbatim French quote of at most 15 words that shows it; "" if absent. Write the quote before you decide.
- present: true or false."""

SCHEMA = Form.model_json_schema()
CONFIG = hashlib.sha256((INSTRUCTIONS + json.dumps(SCHEMA, sort_keys=True)).encode()).hexdigest()[:8]
print(len(INSTRUCTIONS.split()), "words of instructions | config", CONFIG)

### Claude

In [ ]:
import anthropic

assert get_secret("ANTHROPIC_API_KEY"), "No key: add ANTHROPIC_API_KEY in Colab Secrets (key icon), Notebook access on"
claude = anthropic.Anthropic(api_key=get_secret("ANTHROPIC_API_KEY"), max_retries=5)


def extract_claude(text):
    r = claude.messages.parse(
        model=CLAUDE_MODEL, max_tokens=1500,
        system=[{"type": "text", "text": INSTRUCTIONS, "cache_control": {"type": "ephemeral"}}],
        messages=[{"role": "user", "content": text[:MAX_CHARS]}],
        output_format=Form,
        thinking={"type": "disabled"},          # a well-defined task: no thinking, low cost
    )
    u = r.usage
    return {"form": r.parsed_output.model_dump() if r.parsed_output else None, "stop": r.stop_reason,
            "model_returned": r.model, "in": u.input_tokens, "out": u.output_tokens,
            "cache_read": u.cache_read_input_tokens or 0, "cache_write": u.cache_creation_input_tokens or 0}

One document. Read everything that comes back before you run 300 of them, and compare with the catalogue record.

In [ ]:
first = extract_claude(TEXT[KEEP[0]])
first

In [ ]:
CAT.loc[KEEP[0], ["nom", "prenom", "sexe", "profession", "mandat", "soutien", "liste", "supp_nom"]].to_dict()

In [ ]:
def run(fn, name, workers=4):
    """Every document once. Answers are appended to a JSONL file as they arrive: a crash or a rerun never pays twice.
    The file name carries the model and a hash of the instructions and schema: change them, and you get fresh answers."""
    path = LLM / f"{name}_{CONFIG}.jsonl"
    done = {}
    if path.exists():
        for line in path.read_text().splitlines():
            rec = json.loads(line)
            if rec.get("form"):
                done[rec["doc_id"]] = rec
    todo = [i for i in KEEP if i not in done]

    def one(i):
        t0 = time.time()
        try:
            rec = fn(TEXT[i])
        except Exception as e:
            rec = {"form": None, "error": repr(e)[:300]}
        rec.update(doc_id=i, seconds=round(time.time() - t0, 2),
                   utc=datetime.datetime.now(datetime.timezone.utc).isoformat())
        return rec

    with ThreadPoolExecutor(workers) as pool, path.open("a") as f:
        for k, rec in enumerate(pool.map(one, todo), 1):
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
            done[rec["doc_id"]] = rec
            if k % 50 == 0:
                print(f"  {name}: {k}/{len(todo)}")
    out = pd.DataFrame([done.get(i, {"doc_id": i, "form": None}) for i in KEEP]).set_index("doc_id")
    n_err = out.form.isna().sum()
    print(f"{name}: {len(out) - n_err}/{len(out)} forms" + (f", {n_err} errors: rerun the cell" if n_err else ""))
    return out


def cost(res):
    tok = res[["in", "out", "cache_read", "cache_write"]].fillna(0).sum()
    return round(sum(tok[k] * PRICE[k] for k in PRICE) / 1e6, 3)

In [ ]:
RESULTS = {"claude": run(extract_claude, "claude_" + CLAUDE_MODEL)}
print("cost: $", cost(RESULTS["claude"]))

### GPT: the same form, the same instructions

In [ ]:
from openai import OpenAI

gpt = OpenAI(api_key=get_secret("OPENAI_API_KEY") or "missing", max_retries=5)


def extract_gpt(text):
    r = gpt.responses.parse(model=GPT_MODEL, max_output_tokens=1500, instructions=INSTRUCTIONS,
                            input=text[:MAX_CHARS], text_format=Form, reasoning={"effort": "none"})
    u = r.usage
    cached = getattr(getattr(u, "input_tokens_details", None), "cached_tokens", 0) or 0
    return {"form": r.output_parsed.model_dump() if r.output_parsed else None, "stop": r.status,
            "model_returned": r.model, "in": u.input_tokens - cached, "out": u.output_tokens,
            "cache_read": cached, "cache_write": 0}


if get_secret("OPENAI_API_KEY"):
    RESULTS["gpt"] = run(extract_gpt, "gpt_" + GPT_MODEL)
    print("cost: $", cost(RESULTS["gpt"]))
else:
    print("No OPENAI_API_KEY: GPT skipped. Everything below works with Claude alone.")

### The table

In [ ]:
def tidy(res):
    rows = {}
    for i, f in res.form.items():
        if not f:
            continue
        t, s = f["titulaire"], f["suppleant"]
        rows[i] = {"nom": t["nom"], "prenom": t["prenom"], "sexe": t["sexe"], "profession": t["profession"],
                   "mandat": " / ".join(t["mandats_en_cours"]), "soutien": f["soutien"],
                   "supp_nom": s["nom"], "supp_profession": s["profession"],
                   "local_offer": int(f["local_offer"]["present"]), "quote": f["local_offer"]["quote"]}
    return pd.DataFrame.from_dict(rows, orient="index")


EXT = {m: tidy(r) for m, r in RESULTS.items()}
EXT["claude"].head()

### Field by field, against the catalogue

Five outcomes per field: **match**, **mismatch**, **model only** (the catalogue says NR, the model gives a value: an invention, or a catalogue gap?), **catalogue only** (the model left it empty), **both empty**. Names must be equal; professions and parties may differ in wording ("ouvrier surface" and "Ouvrier de la surface"; "PCF" and "Parti communiste français").

In [ ]:
STOP = {"de", "la", "le", "les", "des", "du", "d", "l", "et", "en", "au", "aux", "a"}


def tokens(s):
    return {w for w in norm(s).replace("'", " ").split() if w not in STOP}


def acronym(s):
    return "".join(w[0] for w in norm(s).replace("'", " ").split() if w not in STOP)


def same(a, b, field):
    if field in ("nom", "prenom", "sexe", "supp_nom"):
        a, b = norm(a), norm(b)
        return a == b or (min(len(a), len(b)) > 3 and (a in b or b in a))
    ta, tb = tokens(a), tokens(b)
    if not ta or not tb:
        return False
    return ta <= tb or tb <= ta or len(ta & tb) / len(ta | tb) >= 0.5 or acronym(a) in tb or acronym(b) in ta


def outcome(a, refs, field):
    refs = [b for b in refs if not empty(b)]
    if empty(a):
        return "catalogue only" if refs else "both empty"
    if not refs:
        return "model only"
    return "match" if any(same(a, b, field) for b in refs) else "mismatch"


def compare(ext, field):
    refs = (lambda i: [CAT.at[i, "soutien"], CAT.at[i, "liste"]]) if field == "soutien" else (lambda i: [CAT.at[i, field]])
    return pd.Series({i: outcome(ext.at[i, field], refs(i), field) for i in ext.index})


CHECKED = ["nom", "prenom", "sexe", "profession", "soutien", "supp_nom", "supp_profession"]
OUT = {m: pd.DataFrame({f: compare(e, f) for f in CHECKED}) for m, e in EXT.items()}
TABLE = pd.concat({m: o.apply(lambda c: c.value_counts()).fillna(0).astype(int) for m, o in OUT.items()})
TABLE

In [ ]:
# Accuracy where both have a value
pd.DataFrame({m: (o == "match").sum() / o.isin(["match", "mismatch"]).sum() for m, o in OUT.items()}).round(2)

**Read five disagreements.** Who is right: the model, or the catalogue? Open the PDF.

In [ ]:
def disagreements(model, k=5):
    o = OUT[model].stack()
    o = o[o.isin(["mismatch", "model only"])]
    cat = lambda i, f: f"{CAT.at[i, 'soutien']} | {CAT.at[i, 'liste']}" if f == "soutien" else CAT.at[i, f]
    rows = [{"doc_id": i, "field": f, "outcome": v, "model": EXT[model].at[i, f],
             "catalogue": cat(i, f), "pdf": f"https://archive.org/details/{i}"}
            for (i, f), v in o.sample(min(k, len(o)), random_state=SEED).items()]
    return pd.DataFrame(rows)


disagreements("claude")

The offices: the model's list turned into the same 0/1 as the catalogue's, and crossed with it.

In [ ]:
for m, e in EXT.items():
    x = e.mandat.map(is_local).rename(m)
    print(pd.crosstab(CAT.local_mandate.reindex(x.index).rename("catalogue"), x), "\n")

## 3. The table, and keep what a referee would ask for

One row per candidate: where they ran, what the catalogue says, and what each model extracted. Merge it with election results on *département* and constituency.

In [ ]:
DATA = CAT[["departement", "circonscription", "nom", "prenom", "sexe", "profession", "local_mandate", "deputy", "soutien"]].copy()
for m, e in EXT.items():
    DATA[m + "_mandate"] = e.mandat.map(is_local).reindex(DATA.index)
    DATA[m + "_local_offer"] = e.local_offer.reindex(DATA.index)
DATA.head()

A first look at the question for Session 5: do office holders make more local promises, according to each model?

In [ ]:
DATA.groupby("local_mandate")[[c for c in DATA if c.endswith("_local_offer")]].mean().round(2)

The bundle: the variables (no text), every raw answer, the instructions, the schema, the parameters.

In [ ]:
(RUN / "instructions.txt").write_text(INSTRUCTIONS)
(RUN / "schema.json").write_text(json.dumps(SCHEMA, indent=2))
for f in LLM.glob(f"*_{CONFIG}.jsonl"):
    shutil.copy(f, RUN / f.name)
DATA.drop(columns=["nom", "prenom"]).to_csv(RUN / "dataset.csv")
TABLE.to_csv(RUN / "extraction_vs_catalogue.csv")
params = {
    "run_id": RUN_ID, "date_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "data": {"source": "Archelec (Sciences Po, CEVIPOF) via Internet Archive", "query": QUERY,
             "numFound": FOUND["numFound"], "sample": N, "kept": len(KEEP), "seed": SEED,
             "ids_sha256": hashlib.sha256(json.dumps(KEEP).encode()).hexdigest()},
    "llm": {"models": {m: CLAUDE_MODEL if m == "claude" else GPT_MODEL for m in RESULTS},
            "thinking": "disabled / reasoning effort none", "max_chars": MAX_CHARS,
            "instructions_sha256": hashlib.sha256(INSTRUCTIONS.encode()).hexdigest(),
            "config": CONFIG, "cost_usd": {m: cost(r) for m, r in RESULTS.items()}},
}
(RUN / "params.json").write_text(json.dumps(params, indent=2, ensure_ascii=False))
print(sorted(p.name for p in RUN.iterdir()))

In [ ]:
shutil.make_archive(str(RUN), "zip", RUN)
try:
    from google.colab import files
    files.download(f"{RUN}.zip")
except ImportError:
    print("Bundle:", f"{RUN}.zip")

**What you just did.** A data API with its checks; a form filled in by two models; every field checked against someone else's coding, with the disagreements read against the scan; and a table you can merge. Prompt 13 in `template/prompts/` does the same on your documents, with your agent. Session 5: from the `local_offer` box to a coefficient a referee can accept.